# IMDB Sentiment Analysis — 02: Text Preprocessing & Data Split

**Reconess AI Internship · Phase II · Hands-on Sentiment Analysis**  
**Dataset:** IMDB Dataset of 50K Movie Reviews  
**Notebook goal:** Prepare clean, labeled reviews for the upcoming Bag of Words (BoW) and TF-IDF experiments.

This notebook documents the complete baseline preprocessing workflow, from raw text inspection through the stratified train/test split. It retains the original experiments and saved outputs from the development session.

**Workflow:** Load data → Remove exact duplicates → Inspect raw text → Remove HTML → Lowercase → Expand negations → Remove punctuation → Normalize spaces → Check cleaned text → Split train/test.

> **Reproducibility:** The file path below points to the original local Windows project directory. Update it when moving this project to a different computer. The code outputs shown here were saved from the original run; rerun the notebook from top to bottom when changing preprocessing logic.

## 1. Load the IMDB dataset

Use **Pandas** to read the CSV. The original dataset contains **50,000 movie reviews** with two columns: `review` (input text) and `sentiment` (`positive` or `negative`, the label to predict).

In [10]:
import pandas as pd

df = pd.read_csv("D:/imdb-sentiment-analysis/data/IMDB Dataset.csv")
print("Data shape:", df.shape)
print("Columns:", df.columns)

Data shape: (50000, 2)
Columns: Index(['review', 'sentiment'], dtype='str')


## 2. Remove exact duplicate rows

The exploratory analysis notebook found **418 identical rows**. Removing them before splitting reduces the risk of an identical example appearing in both training and test data.

We store the result in `df_clean` so the original `df` remains available for comparison. **Expected shape:** `(49582, 2)`.

In [11]:
# EDA found 418 identical rows in the original IMDB dataset.
# Remove exact duplicates while preserving a copy in `df`.
df_clean = df.drop_duplicates().reset_index(drop=True)
print("Cleaned data shape:", df_clean.shape)

Cleaned data shape: (49582, 2)


## 3. Inspect a raw review

Examine a complete example to decide which cleaning operations are justified. This review includes HTML line-break tags such as `<br />`, mixed capitalization, punctuation, and the negative contraction `didn't`.

**Why inspect first?** In sentiment classification, removing text without understanding it can accidentally remove important cues such as `not` or `never`.

In [12]:
# Inspect a complete raw review, rather than a truncated DataFrame preview.
# iloc[10] is the 11th review (Python indexes start at zero).
print(df_clean["review"].iloc[10])

Phil the Alien is one of those quirky films where the humour is based around the oddness of everything rather than actual punchlines.<br /><br />At first it was very odd and pretty funny but as the movie progressed I didn't find the jokes or oddness funny anymore.<br /><br />Its a low budget film (thats never a problem in itself), there were some pretty interesting characters, but eventually I just lost interest.<br /><br />I imagine this film would appeal to a stoner who is currently partaking.<br /><br />For something similar but better try "Brother from another planet"


## 4. Remove HTML tags and convert to lowercase

`remove_html_tags()` replaces simple HTML tags with spaces, preserving a boundary between neighboring words. Lowercasing then normalizes variations like `Movie` and `movie`.

**Implementation note:** The simple regex `<.*?>` is suitable for the basic tags observed here but is not a complete HTML parser.

In [13]:
import re


def remove_html_tags(text):
    """Remove basic HTML tags from a review."""
    clean = re.compile(r"<.*?>")
    return re.sub(clean, " ", text)


# Test HTML removal and lowercasing on a single review.
without_tag = remove_html_tags(df_clean["review"].iloc[10]).lower()
print(without_tag)

phil the alien is one of those quirky films where the humour is based around the oddness of everything rather than actual punchlines.  at first it was very odd and pretty funny but as the movie progressed i didn't find the jokes or oddness funny anymore.  its a low budget film (thats never a problem in itself), there were some pretty interesting characters, but eventually i just lost interest.  i imagine this film would appeal to a stoner who is currently partaking.  for something similar but better try "brother from another planet"


## 5. Expand negative contractions

Negative contractions can be useful sentiment signals: `didn't` → `did not`, `won't` → `will not`, `isn't` → `is not`. Expanding these **before** punctuation removal keeps the negation visible to the classifier.

This function handles a selected list of common forms. It uses literal string replacement, so it does not cover every possible English contraction or account for every word-boundary edge case.

In [14]:
def expand_negations(text):
    """Expand a selected set of English negative contractions."""
    negations = [
        "isn't", "aren't", "wasn't", "weren't",
        "don't", "doesn't", "didn't", "won't",
        "wouldn't", "can't", "couldn't",
        "mustn't", "needn't", "shan't",
        "shouldn't", "hasn't", "haven't", "hadn't",
    ]

    negations_replace = [
        "is not", "are not", "was not", "were not",
        "do not", "does not", "did not", "will not",
        "would not", "can not", "could not",
        "must not", "need not", "shall not",
        "should not", "has not", "have not", "had not",
    ]

    for i, neg in enumerate(negations):
        text = text.replace(neg, negations_replace[i])
    return text


# Test the negation expansion on the same lowercase example.
expanded = expand_negations(remove_html_tags(without_tag))
print(expanded)

phil the alien is one of those quirky films where the humour is based around the oddness of everything rather than actual punchlines.  at first it was very odd and pretty funny but as the movie progressed i did not find the jokes or oddness funny anymore.  its a low budget film (thats never a problem in itself), there were some pretty interesting characters, but eventually i just lost interest.  i imagine this film would appeal to a stoner who is currently partaking.  for something similar but better try "brother from another planet"


## 6. Remove punctuation and normalize whitespace

The current experiment replaces punctuation with spaces using `re.sub(r"[^\w\s]", " ", text)`, preserving boundaries between neighboring words. The following `split()` / `join()` step consolidates repeated whitespace into single spaces.

**Current behavior:** Punctuation is replaced with a space, not deleted. The whitespace normalization below then removes repeated spaces. The implementation and saved outputs are preserved.

In [15]:
def remove_punctuation(text):
    """Remove punctuation from a string (current baseline behavior)."""
    return re.sub(r"[^\w\s]", " ", text)


# Inspect punctuation removal and whitespace normalization separately.
punctuated = remove_punctuation(expanded)
print(punctuated)

words = punctuated.split()
print(words)
normalized_text = " ".join(words)
print(normalized_text)

phil the alien is one of those quirky films where the humour is based around the oddness of everything rather than actual punchlines   at first it was very odd and pretty funny but as the movie progressed i did not find the jokes or oddness funny anymore   its a low budget film  thats never a problem in itself   there were some pretty interesting characters  but eventually i just lost interest   i imagine this film would appeal to a stoner who is currently partaking   for something similar but better try  brother from another planet 
['phil', 'the', 'alien', 'is', 'one', 'of', 'those', 'quirky', 'films', 'where', 'the', 'humour', 'is', 'based', 'around', 'the', 'oddness', 'of', 'everything', 'rather', 'than', 'actual', 'punchlines', 'at', 'first', 'it', 'was', 'very', 'odd', 'and', 'pretty', 'funny', 'but', 'as', 'the', 'movie', 'progressed', 'i', 'did', 'not', 'find', 'the', 'jokes', 'or', 'oddness', 'funny', 'anymore', 'its', 'a', 'low', 'budget', 'film', 'thats', 'never', 'a', 'prob

## 7. NLTK experiment and reusable preprocessing pipeline

NLTK's `WordNetLemmatizer` was explored separately during development. Lemmatization can reduce word variants (`movies` → `movie`), but an incorrect part-of-speech assumption can produce unintended changes (e.g., the default noun treatment of `was`).

**Decision for this baseline:** The main `preprocess_text()` function intentionally **does not apply lemmatization**. It combines HTML removal, lowercasing, negation expansion, punctuation removal and whitespace normalization. The original `review` column is retained beside the newly created `preprocessed_review` column.

The `stopwords` resource is downloaded but **stopword removal is not applied**. In sentiment analysis, negation words such as `not` and `never` should not be removed blindly.

In [16]:
import nltk
from nltk.stem import WordNetLemmatizer

# Download resources used during preprocessing exploration.
nltk.download("stopwords")
nltk.download("wordnet")

lemmatizer = WordNetLemmatizer()


def preprocess_text(text):
    """Apply the current baseline cleaning steps to one review."""
    text = remove_html_tags(text)
    text = text.lower()
    text = expand_negations(text)
    text = remove_punctuation(text)
    words = text.split()

    # The lemmatization experiment is intentionally not part of this baseline.
    # lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    # return " ".join(lemmatized_words)
    return " ".join(words)


# Preserve the original review and store the processed version separately.
df_clean["preprocessed_review"] = df_clean["review"].apply(preprocess_text)
print(df_clean["preprocessed_review"].iloc[10])

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User0\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\User0\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


phil the alien is one of those quirky films where the humour is based around the oddness of everything rather than actual punchlines at first it was very odd and pretty funny but as the movie progressed i did not find the jokes or oddness funny anymore its a low budget film thats never a problem in itself there were some pretty interesting characters but eventually i just lost interest i imagine this film would appeal to a stoner who is currently partaking for something similar but better try brother from another planet


## 8. Verify the cleaned dataset and remove new duplicates

Cleaning can cause originally different reviews to become identical after normalization. We check missing values, full-row duplicates, repeated **processed** text, contradictory labels and empty strings, then remove duplicate processed reviews.

**Result of the saved run:** **6 repeated processed reviews**, **0 conflicting-label groups**, and **0 missing values**. After deduplication the final dataset has **49,576 rows** and **3 columns**.

**Empty-text check:** `str.strip() == ""` correctly tests for an empty or whitespace-only string. Checking against `" "` after stripping would not detect an empty string.

In [17]:
# Check data quality before removing duplicates created by cleaning.
print(df_clean.isnull().sum())
print("Duplicated rows:", df_clean.duplicated().sum())
print(
    "Duplicated reviews:",
    df_clean.duplicated(subset="preprocessed_review").sum(),
)
print(
    "Conflicting labels:",
    (df_clean.groupby("preprocessed_review")["sentiment"].nunique() > 1).sum(),
)
print(
    "Empty reviews:",
    (df_clean["preprocessed_review"].str.strip() == "").sum(),
)

# Each duplicate processed text has a consistent label in this run.
df_clean_new = df_clean.drop_duplicates(
    subset="preprocessed_review"
).reset_index(drop=True)

# Repeat the checks after final deduplication.
print(df_clean_new.isnull().sum())
print("Duplicated rows:", df_clean_new.duplicated().sum())
print(
    "Duplicated reviews:",
    df_clean_new.duplicated(subset="preprocessed_review").sum(),
)
print(
    "Conflicting labels:",
    (df_clean_new.groupby("preprocessed_review")["sentiment"].nunique() > 1).sum(),
)
print(
    "Empty reviews:",
    (df_clean_new["preprocessed_review"].str.strip() == "").sum(),
)

review                 0
sentiment              0
preprocessed_review    0
dtype: int64
Duplicated rows: 0
Duplicated reviews: 6
Conflicting labels: 0
Empty reviews: 0
review                 0
sentiment              0
preprocessed_review    0
dtype: int64
Duplicated rows: 0
Duplicated reviews: 0
Conflicting labels: 0
Empty reviews: 0


## 9. Stratified train/test split (80/20)

Define `X` as cleaned review text and `y` as sentiment labels, then use `train_test_split` with:

- `test_size=0.2`: reserve 20% for the final test set.
- `random_state=42`: make the split reproducible.
- `stratify=y`: preserve approximately the same positive/negative distribution in both sets.

**Important anti-leakage rule:** In the next notebook, any learned vectorizer (`CountVectorizer`, `TfidfVectorizer`) must be **fitted only on `X_train`**. Transform `X_test` using the already-fitted vectorizer. Keep the test set out of model selection and hyperparameter tuning.

In [18]:
from sklearn.model_selection import train_test_split

X = df_clean_new["preprocessed_review"]
y = df_clean_new["sentiment"]

# Use stratification to keep positive/negative proportions nearly identical.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(df_clean_new.shape, X_train.shape, X_test.shape, y_train.shape, y_test.shape)
print(y_train.value_counts())
print(y_test.value_counts())


train_df=pd.DataFrame({"review": X_train, "sentiment": y_train}).reset_index(drop=True)
test_df=pd.DataFrame({"review": X_test, "sentiment": y_test}).reset_index(drop=True)

train_df.to_csv("D:/imdb-sentiment-analysis/data/train.csv", index=False)
test_df.to_csv("D:/imdb-sentiment-analysis/data/test.csv", index=False)

print("Train data train shape:",train_df.shape)
print("Test data test shape:",test_df.shape)



(49576, 3) (39660,) (9916,) (39660,) (9916,)
sentiment
positive    19904
negative    19756
Name: count, dtype: int64
sentiment
positive    4977
negative    4939
Name: count, dtype: int64
Train data train shape: (39660, 2)
Test data test shape: (9916, 2)


## 10. Text Preprocessing — Summary

| Step | Result / reason |
|---|---|
| Initial dataset | **50,000** reviews; columns: `review`, `sentiment` |
| Exact duplicates removed | **418**; resulting dataset: **49,582** rows |
| Text normalization | HTML removal, lowercase, selected negative-contraction expansion, punctuation removal, whitespace normalization |
| Lemmatization | Explored with NLTK; deliberately **not included** in the baseline pipeline |
| Quality checks after preprocessing | **0** nulls, **0** conflicting-label groups, **0** empty reviews reported; **6** repeated cleaned reviews before final deduplication |
| Final dataset | **49,576** unique preprocessed reviews |
| Train/test split | **39,660** training reviews and **9,916** testing reviews |

**Label counts after the split:**

| Sentiment | Training | Testing |
|---|---:|---:|
| Positive | 19,904 | 4,977 |
| Negative | 19,756 | 4,939 |

Both subsets remain nearly balanced because of `stratify=y`.

### Key takeaways

The preprocessing pipeline is intentionally simple and reproducible. The dataset is ready for the first traditional machine-learning baseline, with the cleaned text and labels separated into training and testing subsets. Cleaning and deterministic duplicate removal were completed before the split; **vectorizer vocabulary and model parameters must still be learned from training data only**.

### Next notebook: BoW vs TF-IDF

Fit BoW and TF-IDF on the training texts, transform a held-out set without fitting it, and train a baseline classifier (e.g., Logistic Regression). Later compare the baseline with additional experiments, including the optional lemmatization variation. Record all model metrics and changes in the project README.

**For later improvement:** Test contraction coverage and word-boundary edge cases before drawing final performance conclusions. If the preprocessing definition changes, regenerate the dataset and split consistently and document the change.